# 06 - Validation Strategy and Baseline Models

## Objective

1. Choose how we split the data.
2. Train simple baselines and see how far they go.
3. Check which feature groups help (core, destination history, hour of day).

We only use the **validation** set here. The test set is not touched until the final model.

Decisions from notebook 05:
- Fraud only exists in `CASH_OUT` and `TRANSFER`, so we model only these two types.
- `step`, `day_index`, `orig_*`, `dest_unique_origins_before` and `empties_balance_flag` are not used.
- Two new features say how close the amount is to the full balance: `ratio_gap_to_one` and `ratio_is_one`.

## Imports and Data Loading

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

sys.path.append('..')
from src.features import config
from src.data.splits import temporal_masks, TRAIN_END_STEP, VAL_END_STEP
from src.models.feature_sets import FEATURE_SETS, NOT_USED, add_ratio_features, prepare_linear
from src.models.evaluation import evaluate

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
SEED = config.RANDOM_SEED

In [2]:
table = pd.read_parquet(config.PROCESSED_PATH)
table = add_ratio_features(table)
print(table.shape)

(6362620, 30)


## 1. Scope: only CASH_OUT and TRANSFER

We check again that no fraud exists in the other types, then keep only the two types.

In [3]:
in_scope = (table['type_CASH_OUT'] + table['type_TRANSFER']) == 1
print('Fraud outside CASH_OUT / TRANSFER:', int(table.loc[~in_scope, 'isFraud'].sum()),
      f'(out of {(~in_scope).sum():,} rows)')

data = table[in_scope].reset_index(drop=True)
print(f'In scope: {len(data):,} rows, {int(data["isFraud"].sum()):,} frauds, '
      f'{data["isFraud"].mean() * 100:.3f}% fraud')
del table

Fraud outside CASH_OUT / TRANSFER: 0 (out of 3,592,211 rows)
In scope: 2,770,409 rows, 8,213 frauds, 0.296% fraud


## 2. Time-based split

Fraud detection works on future transactions, so we train on the past and validate on the later steps. We use:
- train: step <= 500
- validation: step 501 to 620
- test: step 621 to 743 (not used in this notebook)

In [4]:
tr_mask, va_mask, te_mask = temporal_masks(data, TRAIN_END_STEP, VAL_END_STEP)
split_table = pd.DataFrame({
    'rows': [tr_mask.sum(), va_mask.sum(), te_mask.sum()],
    'frauds': [data.loc[m, 'isFraud'].sum() for m in (tr_mask, va_mask, te_mask)],
}, index=['train', 'validation', 'test'])
split_table['fraud_rate_pct'] = (split_table['frauds'] / split_table['rows'] * 100).round(3)
split_table

,rows,frauds,fraud_rate_pct
train,2645779,5561,0.210
validation,86155,1286,1.493
test,38475,1366,3.550


**Observation:** There are about 11 frauds per step, so validation and test still have 1,286 and 1,366 frauds, even with fewer rows. But the fraud rate goes up: 0.21% in train, 1.49% in validation, 3.55% in test. This is the volume drop after step 400.

Because of this, PR-AUC and alerts are not comparable between splits. We also look at recall at a fixed precision. The test set is small (38k rows), so we will use it only once, at the end.

In [5]:
tr, va = data[tr_mask], data[va_mask]
ytr, yva = tr['isFraud'].to_numpy(), va['isFraud'].to_numpy()
print(len(tr), len(va))

2645779 86155


## 3. Metrics

Accuracy is not useful here. We use:
- **PR-AUC**: main metric, works well with rare fraud.
- **ROC-AUC**: for reference.
- **Best F1** on the validation set.
- **Recall at 80% and 95% precision**: how much fraud we catch if we want few false alerts.
- **Alerts per 10,000**: how many transactions we flag at 95% precision.

The fraud rate is different in each split (see above), and PR-AUC depends on it. So we always look at recall at a fixed precision too.

In [6]:
results = []

def run(model, name, set_name, linear=False):
    cols = FEATURE_SETS[set_name]
    Xtr, Xva = tr[cols], va[cols]
    if linear:
        Xtr, Xva = prepare_linear(Xtr), prepare_linear(Xva)
    start = time.time()
    model.fit(Xtr, ytr)
    secs = time.time() - start
    scores = model.predict_proba(Xva)[:, 1]
    row = {'model': name, 'features': set_name, 'n_features': len(cols), 'seconds': round(secs, 1)}
    row.update(evaluate(yva, scores))
    results.append(row)
    return model, scores

## 4. Baselines

Feature set `S2_core_dest` (core + destination history), no hour of day.

In [7]:
run(DummyClassifier(strategy='prior'), 'Majority (prior)', 'S2_core_dest')

logreg = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=300, class_weight='balanced'))
run(logreg, 'Logistic Regression', 'S2_core_dest', linear=True)

tree = DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, random_state=SEED)
tree, tree_scores = run(tree, 'Decision Tree (depth 6)', 'S2_core_dest')

pd.DataFrame(results).round(4)

,model,features,n_features,seconds,prevalence_pct,pr_auc,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95
0,Majority (prior),S2_core_dest,14,0.1,1.493,0.0149,0.5,0.0294,0.0000,0.0000,0.0000
1,Logistic Regression,S2_core_dest,14,5.7,1.493,0.9997,1.0,0.9992,0.9992,0.9984,156.8104
2,Decision Tree (depth 6),S2_core_dest,14,17.8,1.493,1.0000,1.0,0.9988,1.0000,1.0000,149.8462


**Observation:** The majority baseline has a PR-AUC of 0.015, which is just the fraud rate. Logistic Regression gets 0.9997 and a tree of depth 6 gets 1.0000.

This is too good for a fraud problem. A very simple model already finds almost every fraud, so we must check why before trusting it.

## 5. Which feature groups help?

We compare three feature sets with a Decision Tree and with a Gradient Boosting model (`HistGradientBoostingClassifier` from scikit-learn). Gradient Boosting is only used here to see what the features can give. The real model comparison is the next notebook.

- `S1_core`: transaction features only
- `S2_core_dest`: core + destination history
- `S3_core_dest_hour`: S2 + hour of day

In [8]:
hgb_scores = {}
for set_name in FEATURE_SETS:
    tree_m = DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, random_state=SEED)
    run(tree_m, 'Decision Tree (depth 6)', set_name)
    hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
    _, hgb_scores[set_name] = run(hgb, 'Gradient Boosting', set_name)

ablation = pd.DataFrame(results)
ablation = ablation[ablation['model'] != 'Majority (prior)']
ablation.sort_values(['model', 'features'])[
    ['model', 'features', 'pr_auc', 'roc_auc', 'best_f1', 'recall_at_p80', 'recall_at_p95',
     'alerts_per_10k_at_p95', 'seconds']].round(4)

,model,features,pr_auc,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95,seconds
3,Decision Tree (depth 6),S1_core,1.0000,1.0000,0.9988,1.0000,1.0000,149.8462,10.9
2,Decision Tree (depth 6),S2_core_dest,1.0000,1.0000,0.9988,1.0000,1.0000,149.8462,17.8
5,Decision Tree (depth 6),S2_core_dest,1.0000,1.0000,0.9988,1.0000,1.0000,149.8462,17.4
7,Decision Tree (depth 6),S3_core_dest_hour,1.0000,1.0000,0.9988,1.0000,1.0000,150.6587,22.9
4,Gradient Boosting,S1_core,0.9999,1.0000,0.9988,1.0000,0.9984,151.8194,3.2
6,Gradient Boosting,S2_core_dest,0.9960,0.9976,0.9977,0.9961,0.9961,156.1140,5.0
8,Gradient Boosting,S3_core_dest_hour,0.9724,0.9736,0.9854,0.9736,0.9736,152.8640,9.0
1,Logistic Regression,S2_core_dest,0.9997,1.0000,0.9992,0.9992,0.9984,156.8104,5.7


**Observation:**
- The Decision Tree gives the same result on all three sets (PR-AUC 1.0000). It never needs the destination or hour features.
- Gradient Boosting gets worse when we add features: 0.9999 (S1), 0.9960 (S2), 0.9724 (S3).
- The hour feature hurts the most. The fraud pattern by hour comes from the simulation, so it probably does not hold in the validation period. We drop `hour_of_day`.
- Destination history adds nothing on this easy task. It may matter when the strong feature is removed (next notebook).
- Differences like 0.9999 vs 1.0000 are 1 or 2 frauds out of 1,286. This is noise.

### Decision Tree importance (S2)

In [9]:
imp = pd.Series(tree.feature_importances_, index=FEATURE_SETS['S2_core_dest']).sort_values(ascending=False)
imp.round(4).to_frame('importance')

,importance
ratio_is_one,0.9874
oldbalanceOrg_log,0.0126
amount_log,0.0001
amount_to_balance_ratio,0.0000
dest_steps_since_last,0.0000
oldbalanceDest_log,0.0000
ratio_gap_to_one,0.0000
zero_old_balance_orig,0.0000
dest_txn_count_before,0.0000
type_TRANSFER,0.0000


**Observation:** `ratio_is_one` has 98.7% of the importance and `oldbalanceOrg_log` has 1.3%. All the other features are at 0. The tree is almost one single rule.

## 6. Does the split matter?

Same Gradient Boosting model and same features (S2), but with a random split of the train and validation rows instead of a time split. If the random split looks better, it would give a too optimistic idea of the model.

In [10]:
pool = data[tr_mask | va_mask]
cols = FEATURE_SETS['S2_core_dest']
Xa, Xb, ya, yb = train_test_split(pool[cols], pool['isFraud'], test_size=0.25,
                                  stratify=pool['isFraud'], random_state=SEED)
rand_model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
rand_model.fit(Xa, ya)
rand_res = evaluate(yb, rand_model.predict_proba(Xb)[:, 1])

temporal_res = [r for r in results if r['model'] == 'Gradient Boosting' and r['features'] == 'S2_core_dest'][0]
split_compare = pd.DataFrame([
    {'split': 'time split (validation)', **{k: temporal_res[k] for k in rand_res}},
    {'split': 'random split', **rand_res},
])
split_compare.round(4)

,split,prevalence_pct,pr_auc,roc_auc,best_f1,recall_at_p80,recall_at_p95,alerts_per_10k_at_p95
0,time split (validation),1.493,0.9960,0.9976,0.9977,0.9961,0.9961,156.1140
1,random split,0.251,0.9893,0.9991,0.9930,0.9936,0.9936,26.1939


**Observation:** The random split is not better than the time split (0.9893 vs 0.9960). Here the task is so easy that the split type does not change much. The alerts per 10,000 are not comparable, because the fraud rate is different (0.25% vs 1.49%).

We keep the time split, because it is how the model would be used in real life.

## 7. What does the best model still miss?

At the 95% precision point on the validation set, we look at the missed frauds by type and by amount to balance ratio.

In [11]:
best_set = ablation[ablation['model'] == 'Gradient Boosting'].sort_values('pr_auc').iloc[-1]['features']
print('Best feature set for Gradient Boosting:', best_set)

scores = hgb_scores[best_set]
from sklearn.metrics import precision_recall_curve
p, r, t = precision_recall_curve(yva, scores)
ok = p[:-1] >= 0.95
thr = t[np.argmax(np.where(ok, r[:-1], -1))] if ok.any() else 0.5

pred = scores >= thr
missed = va[(yva == 1) & (~pred)]
caught = va[(yva == 1) & pred]
print(f'Threshold: {thr:.4f}  caught: {len(caught)}  missed: {len(missed)}')

summary = pd.DataFrame({
    'caught': caught[['ratio_is_one', 'dest_is_new', 'type_TRANSFER', 'zero_old_balance_orig']].mean(),
    'missed': missed[['ratio_is_one', 'dest_is_new', 'type_TRANSFER', 'zero_old_balance_orig']].mean(),
})
summary.round(3)

Best feature set for Gradient Boosting: S1_core
Threshold: 0.0009  caught: 1284  missed: 2


,caught,missed
ratio_is_one,0.984,0.0
dest_is_new,0.628,0.0
type_TRANSFER,0.501,0.0
zero_old_balance_orig,0.001,1.0


**Observation:** At 95% precision the model misses only 2 frauds out of 1,286. Both have a zero balance on the origin account, so `ratio_is_one` is 0 for them. They are probably the zero-amount frauds from notebook 03. About 150 alerts per 10,000 is also almost the same as the fraud rate (149), so nearly every alert is a real fraud.

The threshold (0.0009) is not meaningful. We will choose it properly later.

## 8. Conclusion and next step

**Main result.** Models reach almost perfect scores (PR-AUC 0.9997 to 1.0000 with a simple Logistic Regression or a small tree). The Decision Tree puts 98.7% of its importance on one feature: `ratio_is_one`.


**Decisions**
- Drop `hour_of_day` (it made Gradient Boosting worse).
- Keep the time split.
- Do not use PR-AUC alone, and do not choose a model from differences of 0.0001.
- The two missed frauds have a zero balance. We leave them.

**Next notebook:** `07_model_comparison.ipynb`.
1. Test the single rule `ratio_is_one` alone.
2. Remove the ratio features and see what the other features can do.
3. Compare Logistic Regression, Decision Tree, Random Forest and Gradient Boosting on the easy and on the harder feature sets, with confidence intervals.
4. Compare ways to handle the imbalance (none, class weights, undersampling, SMOTE).